In [4]:
import os
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from langchain_openai import ChatOpenAI
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Load environment variables
from dotenv import load_dotenv
import os

# Load environment variables and set OpenAI API key
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

# Set a random seed for reproducibility
np.random.seed(42)

In [5]:
# Generate sample data
n_rows = 1000

# Generate dates
start_date = datetime(2022, 1, 1)
dates = [start_date + timedelta(days=i) for i in range(n_rows)]

# Define data categories
makes = ['Toyota', 'Honda', 'Ford', 'Chevrolet', 'Nissan', 'BMW', 'Mercedes', 'Audi', 'Hyundai', 'Kia']
models = ['Sedan', 'SUV', 'Truck', 'Hatchback', 'Coupe', 'Van']
colors = ['Red', 'Blue', 'Black', 'White', 'Silver', 'Gray', 'Green']

# Create the dataset
data = {
    'Date': dates,
    'Make': np.random.choice(makes, n_rows),
    'Model': np.random.choice(models, n_rows),
    'Color': np.random.choice(colors, n_rows),
    'Year': np.random.randint(2015, 2023, n_rows),
    'Price': np.random.uniform(20000, 80000, n_rows).round(2),
    'Mileage': np.random.uniform(0, 100000, n_rows).round(0),
    'EngineSize': np.random.choice([1.6, 2.0, 2.5, 3.0, 3.5, 4.0], n_rows),
    'FuelEfficiency': np.random.uniform(20, 40, n_rows).round(1),
    'SalesPerson': np.random.choice(['Alice', 'Bob', 'Charlie', 'David', 'Eva'], n_rows)
}

# Create DataFrame and sort by date
df = pd.DataFrame(data).sort_values('Date')

In [6]:
df.head()

,Date,Make,Model,Color,Year,Price,Mileage,EngineSize,FuelEfficiency,SalesPerson
0,2022-01-01,Mercedes,Sedan,Green,2022,57952.65,5522.0,2.0,24.7,Alice
1,2022-01-02,Chevrolet,Hatchback,Red,2021,58668.22,94238.0,1.6,26.2,Bob
2,2022-01-03,Audi,Truck,White,2019,69187.87,7482.0,2.0,28.0,David
3,2022-01-04,Nissan,Hatchback,Black,2016,40004.44,43846.0,3.5,24.8,David
4,2022-01-05,Mercedes,Hatchback,Red,2016,63983.07,52988.0,2.5,24.1,Alice


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Date            1000 non-null   datetime64[us]
 1   Make            1000 non-null   str           
 2   Model           1000 non-null   str           
 3   Color           1000 non-null   str           
 4   Year            1000 non-null   int32         
 5   Price           1000 non-null   float64       
 6   Mileage         1000 non-null   float64       
 7   EngineSize      1000 non-null   float64       
 8   FuelEfficiency  1000 non-null   float64       
 9   SalesPerson     1000 non-null   str           
dtypes: datetime64[us](1), float64(4), int32(1), str(4)
memory usage: 74.3 KB


In [9]:
df.describe()

,Date,Year,Price,Mileage,EngineSize,FuelEfficiency
count,1000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,2023-05-15 12:00:00,2018.445000,51145.360800,48484.643000,2.744500,29.688500
min,2022-01-01 00:00:00,2015.000000,20026.570000,19.000000,1.600000,20.000000
25%,2022-09-07 18:00:00,2017.000000,36859.940000,23191.500000,2.000000,24.500000
50%,2023-05-15 12:00:00,2018.000000,52215.155000,47506.000000,2.500000,29.700000
75%,2024-01-20 06:00:00,2020.000000,65741.147500,73880.250000,3.500000,34.700000
max,2024-09-26 00:00:00,2022.000000,79972.640000,99762.000000,4.000000,40.000000
std,NaN,2.256117,17041.610861,29103.404593,0.839389,5.896316


In [10]:
df.describe(include='object')

C:\Users\Admin\AppData\Local\Temp\ipykernel_18032\87514550.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include='object')


,Make,Model,Color,SalesPerson
count,1000,1000,1000,1000
unique,10,6,7,5
top,Toyota,SUV,White,Alice
freq,118,176,160,216


In [ ]:
agent = create_pandas_dataframe_agent(
    ChatOpenAI(model="gpt-4o-mini", temperature=0),
    df,
    verbose=True,
    allow_dangerous_code=True,
    agent_type="openai-tools",    # ✅ string instead of AgentType
)

print("Data Analysis Agent is ready. You can now ask questions about the data.")

Data Analysis Agent is ready. You can now ask questions about the data.


In [14]:
def ask_agent(question):
    """Function to ask questions to the agent and display the response"""
    response = agent.invoke({
        "input": question,
        "agent_scratchpad": f"Human: {question}\nAI: To answer this question, I need to use Python to analyze the dataframe. I'll use the python_repl_ast tool.\n\nAction: python_repl_ast\nAction Input: ",
    })
    print(f"Question: {question}")
    print(f"Answer: {response}")
    print("---")

In [15]:
ask_agent("What is the average price of cars sold?")
ask_agent("What model is sold the most?")
ask_agent("How make is sold the most?")



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "df['Price'].mean()"}`


51145.360799999995The average price of cars sold is approximately $51,145.36.

> Finished chain.
Question: What is the average price of cars sold?
Answer: {'input': 'What is the average price of cars sold?', 'agent_scratchpad': "Human: What is the average price of cars sold?\nAI: To answer this question, I need to use Python to analyze the dataframe. I'll use the python_repl_ast tool.\n\nAction: python_repl_ast\nAction Input: ", 'output': 'The average price of cars sold is approximately $51,145.36.'}
---


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "df['Model'].mode()"}`


0    SUV
Name: Model, dtype: strThe model that is sold the most is the "SUV".

> Finished chain.
Question: What model is sold the most?
Answer: {'input': 'What model is sold the most?', 'agent_scratchpad': "Human: What model is sold the most?\nAI: To answer this questio